<a href="https://colab.research.google.com/github/tarunmikkilineni/dataviz-exercises-TarunMikkilineni/blob/main/lecture04_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 4 — Class Exercise
## Scatter & Bubble Charts: Gapminder

> **Push to:** `week04/lecture04_exercise.ipynb`

**Rules:**
1. Colour used **sparingly** — one categorical variable, no rainbow
2. If showing all continents, either use accessible palette OR grey all + highlight one
3. `size_max` set when using bubble size
4. Log scale for GDP per capita
5. Insight title

---


In [1]:
import pandas as pd
import plotly.express as px


# Dataset: Gapminder — GDP, Life Expectancy, Population by Country
# Source: Gapminder Foundation (gapminder.org)

df = px.data.gapminder()
print(f"Loaded: {len(df)} rows")
print(df.head())

Loaded: 1704 rows
       country continent  year  lifeExp       pop   gdpPercap iso_alpha  \
0  Afghanistan      Asia  1952   28.801   8425333  779.445314       AFG   
1  Afghanistan      Asia  1957   30.332   9240934  820.853030       AFG   
2  Afghanistan      Asia  1962   31.997  10267083  853.100710       AFG   
3  Afghanistan      Asia  1967   34.020  11537966  836.197138       AFG   
4  Afghanistan      Asia  1972   36.088  13079460  739.981106       AFG   

   iso_num  
0        4  
1        4  
2        4  
3        4  
4        4  


In [2]:
# explore

print(df.info())
print("Years:", sorted(df['year'].unique()))
print("Continents:", df['continent'].unique())
print(df.describe().round(1))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1704 entries, 0 to 1703
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   country    1704 non-null   object 
 1   continent  1704 non-null   object 
 2   year       1704 non-null   int64  
 3   lifeExp    1704 non-null   float64
 4   pop        1704 non-null   int64  
 5   gdpPercap  1704 non-null   float64
 6   iso_alpha  1704 non-null   object 
 7   iso_num    1704 non-null   int64  
dtypes: float64(2), int64(3), object(3)
memory usage: 106.6+ KB
None
Years: [np.int64(1952), np.int64(1957), np.int64(1962), np.int64(1967), np.int64(1972), np.int64(1977), np.int64(1982), np.int64(1987), np.int64(1992), np.int64(1997), np.int64(2002), np.int64(2007)]
Continents: ['Asia' 'Europe' 'Africa' 'Americas' 'Oceania']
         year  lifeExp           pop  gdpPercap  iso_num
count  1704.0   1704.0  1.704000e+03     1704.0   1704.0
mean   1979.5     59.5  2.960121e+07     7215.3    

## Task 1 — Scatter: life expectancy change over time

**What to build:** A scatter showing **GDP per capita vs life expectancy** for **two years** (2002 and 2007) to show how both moved — use **colour for year** (just 2 colours), **one continent only**.

Choose any continent except Africa (that was the example). Highlight the change direction.

> 💡 Filter: `df.loc[df['continent'] == 'YOUR_CHOICE']` then filter years


In [3]:
import numpy as np
import plotly.graph_objects as go

# Filter: one continent (Asia), two years
asia = df.loc[(df['continent'] == 'Asia') & (df['year'].isin([2002, 2007]))].copy()
asia['year'] = asia['year'].astype(str)

# Insight numbers for the title
avg = asia.groupby('year')['lifeExp'].mean()
gain = avg['2007'] - avg['2002']

fig = px.scatter(
    asia, x='gdpPercap', y='lifeExp', color='year',
    log_x=True,
    hover_name='country',
    color_discrete_map={'2002': '#B0B0B0', '2007': '#1F77B4'},  # grey = before, blue = after
    labels={'gdpPercap': 'GDP per capita (US$, log scale)',
            'lifeExp': 'Life expectancy (years)', 'year': 'Year'},
    title=f"Asian countries got richer and lived ~{gain:.1f} years longer on average, 2002 to 2007",
)
fig.update_traces(marker=dict(size=9, opacity=0.85))

# Lines connecting each country's 2002 point to its 2007 point (shows direction of change)
wide = asia.pivot(index='country', columns='year', values=['gdpPercap', 'lifeExp'])
xs, ys = [], []
for c in wide.index:
    xs += [wide.loc[c, ('gdpPercap', '2002')], wide.loc[c, ('gdpPercap', '2007')], None]
    ys += [wide.loc[c, ('lifeExp', '2002')], wide.loc[c, ('lifeExp', '2007')], None]
fig.add_trace(go.Scatter(x=xs, y=ys, mode='lines',
                         line=dict(color='lightgrey', width=1),
                         hoverinfo='skip', showlegend=False))

# Annotate the biggest life-expectancy gainer (log axis -> annotation x must be log10)
change = (wide[('lifeExp', '2007')] - wide[('lifeExp', '2002')])
top = change.idxmax()
fig.add_annotation(
    x=np.log10(wide.loc[top, ('gdpPercap', '2007')]),
    y=wide.loc[top, ('lifeExp', '2007')],
    text=f"{top}: +{change.max():.1f} years",
    showarrow=True, arrowhead=2, ax=-60, ay=-40,
)

fig.update_layout(template='plotly_white', legend_title_text='Year')
fig.show()


## Task 2 — Bubble chart: tell a story

**What to build:** A bubble chart (full 2007 dataset, all countries) where:
- x = GDP per capita (log scale)
- y = life expectancy
- size = population
- colour = ONE continent highlighted (your choice), all others grey
- At least one annotation explaining the highlighted group's story

> This is the grey-and-highlight technique applied to a bubble chart.


In [4]:
df07 = df.loc[df['year'] == 2007].copy()

HIGHLIGHT = 'Africa'
df07['group'] = np.where(df07['continent'] == HIGHLIGHT, HIGHLIGHT, 'Other continents')

# Draw grey bubbles first so the highlighted ones sit on top
df07 = df07.sort_values('group', ascending=False)

fig = px.scatter(
    df07, x='gdpPercap', y='lifeExp', size='pop',
    color='group',
    log_x=True,
    size_max=60,                                   # required by the rules
    hover_name='country',
    color_discrete_map={HIGHLIGHT: '#D55E00', 'Other continents': '#C8C8C8'},
    labels={'gdpPercap': 'GDP per capita (US$, log scale)',
            'lifeExp': 'Life expectancy (years)',
            'group': '', 'pop': 'Population'},
    title="In 2007, African countries sat at the bottom of the life-expectancy ranking at every income level",
)
fig.update_traces(marker=dict(line=dict(width=0.5, color='white'), opacity=0.8))

# Annotation: story of the highlighted group
fig.add_annotation(
    x=np.log10(2500), y=43,
    text="Many African countries have life expectancy<br>below 55, even where income is similar<br>to better-performing countries elsewhere",
    showarrow=True, arrowhead=2, ax=140, ay=20,
    align='left', font=dict(size=12, color='#D55E00'),
)

fig.update_layout(template='plotly_white', legend=dict(orientation='h', y=-0.2))
fig.show()
